# Projeto Integrador — Diagnóstico Preditivo de Risco Financeiro e Falência
## Integrante 1 — Treinamento e Comparação de Modelos Tabulares
**Responsável:** Mateus

### Objetivos:
1. Analisar os dados históricos de empresas polonesas (1 a 5 anos de horizonte).
2. Tratar valores ausentes via imputação por mediana e preparar matriz de atributos.
3. Selecionar atributos financeiros essenciais e analisar correlação com a falência.
4. Treinar e comparar modelos supervisionados: **Random Forest**, **XGBoost** e **Regressão Logística**.
5. Avaliar o desempenho com métricas técnicas: Acurácia, Acurácia Balanceada, Precisão, Recall, F1-Score, ROC-AUC e Matriz de Confusão.
6. Salvar o modelo em formatos de produção (**ONNX** e **Joblib**) e exportar os metadados para integração com o Backend e Frontend.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
import xgboost as xgb
import joblib
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

sns.set_theme(style='whitegrid')

### 1. Carregamento dos Arquivos CSV (1 a 5 anos)
Os dados cobrem empresas ao longo de 5 períodos, totalizando mais de 43.000 observações.

In [ ]:
dados_dir = 'dados'
dfs = []
for ano in range(1, 6):
    path = os.path.join(dados_dir, f'{ano}year.csv')
    df_ano = pd.read_csv(path, na_values='?')
    print(f'{ano}year.csv: {df_ano.shape[0]} empresas x {df_ano.shape[1]} colunas')
    dfs.append(df_ano)

df_full = pd.concat(dfs, ignore_index=True)
print(f'\nTotal Consolidado: {df_full.shape[0]} registros x {df_full.shape[1]} colunas')

### 2. Seleção de Atributos Financeiros e Dicionário
Selecionamos os 10 indicadores contábeis mais consolidados na literatura corporativa e compatíveis com a arquitetura de produção do sistema.

In [ ]:
features = [f'Attr{i}' for i in range(1, 11)]
feature_descriptions = {
    'Attr1': 'Lucro Líquido / Ativo Total',
    'Attr2': 'Passivo Total / Ativo Total',
    'Attr3': 'Capital de Giro / Ativo Total',
    'Attr4': 'Ativo Circulante / Passivo Circulante (Liquidez Corrente)',
    'Attr5': '[(Caixa + Títulos + Recebíveis - Passivo Circulante) / (Despesas Operacionais - Depreciação)] * 365',
    'Attr6': 'Lucros Retidos / Ativo Total',
    'Attr7': 'EBIT / Ativo Total',
    'Attr8': 'Patrimônio Líquido / Passivo Total',
    'Attr9': 'Vendas / Ativo Total',
    'Attr10': 'Patrimônio Líquido / Ativo Total'
}

X = df_full[features].copy().replace([np.inf, -np.inf], np.nan)
y = df_full['class'].astype(int).copy()

# Imputação pela mediana amostral
medians = X.median().round(4).to_dict()
X_imputed = X.fillna(X.median())
print('Medianas Amostrais calculadas:')
for f, m in medians.items():
    print(f'  {f} ({feature_descriptions[f]}): {m}')

### 3. Análise do Desbalanceamento de Classes
A falência é um evento raro (~4.8% da base), exigindo técnicas como pesos balanceados (`class_weight='balanced'`) e `scale_pos_weight`.

In [ ]:
contagem = y.value_counts()
scale_pos_weight = contagem[0] / contagem[1]
print(f'Empresas Saudáveis (Classe 0): {contagem[0]} ({contagem[0]/len(y):.2%})')
print(f'Empresas Falidas (Classe 1):   {contagem[1]} ({contagem[1]/len(y):.2%})')
print(f'Razão de Desbalanceamento:     {scale_pos_weight:.2f}')

plt.figure(figsize=(6, 4))
sns.countplot(x=y, palette=['#1a7f37', '#c62828'])
plt.title('Distribuição de Empresas: Saudáveis (0) vs Falidas (1)')
plt.xlabel('Situação da Empresa')
plt.ylabel('Quantidade de Registros')
plt.show()

### 4. Divisão em Treino e Teste (Estratificada 80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Tamanho do Treino: {X_train.shape[0]} empresas')
print(f'Tamanho do Teste:  {X_test.shape[0]} empresas')

### 5. Treinamento e Comparação dos Modelos (Random Forest, XGBoost e Regressão Logística)

In [ ]:
modelos = {
    'Regressão Logística': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_split=5, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, n_jobs=-1)
}

resultados = {}
for nome, mod in modelos.items():
    mod.fit(X_train, y_train)
    y_pred = mod.predict(X_test)
    y_prob = mod.predict_proba(X_test)[:, 1]
    
    resultados[nome] = {
        'Acurácia': accuracy_score(y_test, y_pred),
        'Acurácia Balanceada': balanced_accuracy_score(y_test, y_pred),
        'Precisão': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

df_res = pd.DataFrame(resultados).T
display(df_res.round(4))

### 6. Curvas ROC e Matriz de Confusão do Random Forest

In [ ]:
rf_model = modelos['Random Forest']
y_rf_pred = rf_model.predict(X_test)
y_rf_prob = rf_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_rf_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Saudável (0)', 'Falida (1)'], yticklabels=['Saudável (0)', 'Falida (1)'])
plt.title('Matriz de Confusão — Random Forest')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.show()

print('Relatório de Classificação Detalhado:')
print(classification_report(y_test, y_rf_pred, target_names=['Saudável', 'Falida']))

### 7. Importância das Variáveis (Feature Importance)

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
labels = [f'{k} ({feature_descriptions[k]})' for k in importances.index]

plt.figure(figsize=(10, 6))
plt.barh(labels, importances.values, color='#2c5282')
plt.title('Importância Relativa dos Indicadores Financeiros (Random Forest)')
plt.xlabel('Importância')
plt.show()

### 8. Exportação do Modelo Treinado (ONNX e Joblib) e Metadados

In [ ]:
# 1. Joblib
joblib.dump(rf_model, 'modelo_risco.joblib')
print('Modelo salvo em modelo_risco.joblib')

# 2. ONNX
initial_type = [('float_input', FloatTensorType([None, len(features)]))]
onnx_model = convert_sklearn(rf_model, initial_types=initial_type, target_opset=12, options={id(rf_model): {'zipmap': False}})
with open('modelo_risco.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())
print('Modelo salvo em modelo_risco.onnx')

# 3. Metadados
meta = {
    'feature_names': features,
    'feature_descriptions': feature_descriptions,
    'medians': medians,
    'threshold_medio_risco': 0.35,
    'threshold_alto_risco': 0.65
}
with open('model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=4, ensure_ascii=False)
print('Metadados exportados com sucesso!')